# eSett Open Data API — quarterly BESS historical backfill

**One calendar-quarter API request and one JSON partition per dataset**,
including all selected Swedish MBAs in the same request. For 2024-07 through
2026-06, that is 8 files per dataset (32 across the four existing eSett
datasets), provided all quarter requests succeed.

Begin with a **one-day smoke test only in memory**, then optionally request one
full quarter before enabling the full two-year backfill. Source records retain
UTC timestamps and MBA labels; they are not treated as measured factory load or
day-ahead market prices.

OpenAPI: https://api.opendata.esett.com/openapi?format=json. No token required
for the separate public Open Data API. Queries use MBA EIC codes. Query-size/row
limits may reject full-quarter requests: this implementation **does not silently
split into daily requests**. Review the failure and the official limits before
changing the window size.


## Notebook dependencies

Run the next cell once to install the BESS package in editable mode in your
notebook kernel, along with pandas. This uses the same code as the CLI and
tests.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "bess_optimizer" / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "bess_optimizer" / "pyproject.toml").exists(), "Launch Jupyter from repo root or notebooks/"


In [ ]:
%pip install -e "{ROOT / 'bess_optimizer'}" pandas

In [ ]:
import json
from datetime import date
import pandas as pd

from bess_optimizer.data.esett_api import (
    ROUTES,
    backfill,
    fetch_day,
    local_day_window,
    mba_options,
    new_session,
    swedish_mba_codes,
    check_day,
    metering_grid_areas,
)

session = new_session()
SAMPLE_DAY = date(2026, 6, 24)
OUTPUT_ROOT = ROOT / "data" / "history" / "source" / "esett"

## 1. Discover Swedish MBA EIC codes

The four datasets can have different option lists. Inspect the returned names
and codes before choosing a geographic scope. `/EXP18/MBAOptions` may have a
nested JSON list.


In [ ]:
all_options = {}
swedish_codes = {}
for dataset in ROUTES:
    all_options[dataset] = mba_options(dataset, session)
    swedish_codes[dataset] = swedish_mba_codes(all_options[dataset])
    print(f"{dataset:18} Swedish EIC codes: {swedish_codes[dataset]}")
    if not swedish_codes[dataset]:
        print("  Could not automatically identify Sweden: inspect raw groups below")
        print(json.dumps(all_options[dataset][:2], indent=2)[:1500])

## 2. One-MBA / one-day smoke test (not a stored partition)

To avoid writing tiny daily files, this smoke test makes a direct one-day
request and displays the response **in memory only**. Quarterly backfill begins
in Section 4. Swedish local days contain 92/96/100 intervals around
daylight-saving transitions. Rows may exceed interval counts for individual
meter-grid areas.


In [ ]:
ONE_MBA = {dataset: codes[:1] for dataset, codes in swedish_codes.items()}
assert all(ONE_MBA.values()), "Inspect /MBAOptions and select Swedish EIC codes manually"
print("Local day UTC interval:", local_day_window(SAMPLE_DAY))
smoke_results = {}
for dataset, codes in ONE_MBA.items():
    try:
        rows = fetch_day(dataset, codes[0], SAMPLE_DAY, session)
        smoke_results[dataset] = rows
        print(dataset, codes[0], check_day(rows, SAMPLE_DAY))
    except Exception as exc:
        smoke_results[dataset] = exc
        print(f"{dataset} failed: {type(exc).__name__}: {exc}")

## 2b. When load-profile API requests exceed provider limits

`/EXP18/LoadProfile` can return many meter-grid areas. If that smoke test fails
with a row-limit error, discover MGA codes via `/EXP03/MeteringGridAreas` and
query a single MGA. This is a **subset**, not the complete Sweden load-profile
dataset.


In [ ]:
load_eic = ONE_MBA["load_profile"][0]
areas = metering_grid_areas(load_eic, session)
display(pd.DataFrame(areas).head(10))

# Uncomment after choosing a real code returned by the discovery request.
# chosen_mga = areas[0]['mgaCode']
# rows = fetch_day('load_profile', load_eic, SAMPLE_DAY, session, mga=chosen_mga)
# print(check_day(rows, SAMPLE_DAY))

## 3. Inspect sampled API records

The values below are the direct one-day responses, **not stored daily
partitions**. Inspect `timestampUTC`, `mba`, and each dataset's market-specific
fields. A 204 or empty response is not complete coverage.


In [ ]:
for dataset, result in smoke_results.items():
    if isinstance(result, Exception):
        continue
    print("\n", dataset)
    frame = pd.DataFrame(result)
    display(frame.head(5))
    print("API columns:", list(frame.columns))

## 4. Full-quarter test, then optional two-year history

**One HTTP request per dataset per quarter** contains repeated `mba` query
parameters for all selected Swedish MBAs. One output file is written per dataset
per calendar quarter (for example, `production/2024-Q3.json`). An API row-limit
or timeout error stops the request without writing a misleading partial
partition. No automatic daily/monthly fallback is used.

The first flag below optionally fetches **2026-Q2** as a live size/capability
test. The second flag requests all 8 quarters from 2024-Q3 through 2026-Q2. Run
the full backfill only after inspecting the provider's response to the quarter
test. Cached partitions are reused when their MBA selection and coverage match;
incomplete partitions are re-requested on resume.

The source may change or revise observations. Set `overwrite=True` to refresh.
The full history covers 2024-07-01 through 2026-06-30 inclusive.


In [ ]:
RUN_ONE_QUARTER = False  # Enable after checking the one-day API schemas
RUN_BACKFILL = False  # Enable only after a successful full-quarter request
FULL_SWEDEN = {dataset: codes for dataset, codes in swedish_codes.items()}
BACKFILL_START = date(2024, 7, 1)
BACKFILL_END_EXCLUSIVE = date(2026, 7, 1)

if RUN_ONE_QUARTER:
    for dataset in ROUTES:
        result = backfill(
            [dataset],
            FULL_SWEDEN,
            date(2026, 4, 1),
            date(2026, 7, 1),
            root=OUTPUT_ROOT,
            session=session,
            pause_seconds=0.3,
        )
        display(pd.DataFrame(result))

if RUN_BACKFILL:
    for dataset in ROUTES:
        result = backfill(
            [dataset],
            FULL_SWEDEN,
            BACKFILL_START,
            BACKFILL_END_EXCLUSIVE,
            root=OUTPUT_ROOT,
            session=session,
            pause_seconds=0.3,
        )
        display(pd.DataFrame(result)[["dataset", "quarter", "status", "row_count", "missing_quarter_hours"]])

## Interpretation and integration

- eSett `production` solar is a **market-area energy profile (MWh)**, not
  measured site PV. Validate source MBAs before normalizing.
- `consumption` and `load_profile` are area- or meter-grid-level observations,
  not factory metering.
- `imbalance_prices` is not Nord Pool day-ahead spot, FCR-N or mFRR reserve
  auctions.
- All selected Swedish MBAs are retained as distinct records **inside each
  quarterly file**, not stored as daily or per-MBA files.
- The notebook produces source-preserving JSON under
  `data/history/source/esett/<dataset>/YYYY-Qn.json`, ignored by Git. The
  existing `data/raw/` representative-day data remains unchanged.
- This is separate from normalized canonical BESS model inputs. Quarterly API
  availability and data completeness are not guaranteed by the source; check
  every quarter audit before modeling.
